# Run the paper comparison from a notebook

This notebook is a thin interactive wrapper around `scripts/run_all_methods.py`. Use it when you want paper-style output tables without leaving Jupyter.

For the main step-by-step explanation of the candidate libraries, use `01_step_by_step_candidate_libraries.ipynb`.


## 0. User controls

The variables below correspond directly to the command-line arguments of `scripts/run_all_methods.py`.

For final paper results, use `profile="paper"` and leave `maxiter_override=None`, `popsize_override=None`.


In [ ]:
# Robust project-root discovery.
# This avoids failures when a Jupyter kernel is started in a directory that is
# later moved/deleted, in which case Path.cwd() itself can raise FileNotFoundError.
import os
import sys
from pathlib import Path


def find_fpde_project_root() -> Path:
    """Return the repository root containing dataset_configs.py and data/.

    Priority:
    1. FPDE_PROJECT_ROOT environment variable, if set.
    2. Current/PWD directories and their parents, if available.
    3. Common local search locations. This keeps notebooks runnable from
       project root, from notebooks/, and after opening a notebook from an IDE.
    """
    def looks_like_root(path: Path) -> bool:
        return (
            (path / "dataset_configs.py").is_file()
            and (path / "weak_pareto_fde_discovery.py").is_file()
            and (path / "data").is_dir()
        )

    candidates = []
    env_root = os.environ.get("FPDE_PROJECT_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())

    # os.getcwd() can fail if the kernel's working directory was deleted.
    try:
        candidates.append(Path(os.getcwd()).expanduser())
    except FileNotFoundError:
        pass

    # PWD may still contain a useful absolute path even when os.getcwd() fails.
    pwd = os.environ.get("PWD")
    if pwd:
        candidates.append(Path(pwd).expanduser())

    # Also try the directory containing this notebook if Jupyter exposes it via env.
    for key in ("NOTEBOOK_DIR", "JUPYTER_SERVER_ROOT"):
        value = os.environ.get(key)
        if value:
            candidates.append(Path(value).expanduser())

    seen = set()
    for cand in candidates:
        try:
            cand = cand.resolve(strict=False)
        except Exception:
            continue
        for path in (cand, cand.parent, *cand.parents):
            if path in seen:
                continue
            seen.add(path)
            if looks_like_root(path):
                return path

    # Conservative bounded search over common project locations.
    search_roots = [
        Path.home() / "Desktop" / "research",
        Path.home() / "Desktop",
        Path.home(),
        Path("/mnt/data"),
    ]
    max_dirs = 5000
    for base in search_roots:
        if not base.exists():
            continue
        visited = 0
        for dirpath, dirnames, filenames in os.walk(base):
            visited += 1
            # Keep the search cheap and avoid hidden/cache directories.
            dirnames[:] = [d for d in dirnames if not d.startswith(".") and d not in {"__pycache__", ".ipynb_checkpoints"}]
            if "dataset_configs.py" in filenames and "weak_pareto_fde_discovery.py" in filenames:
                candidate = Path(dirpath)
                if looks_like_root(candidate):
                    return candidate
            if visited >= max_dirs:
                break

    raise FileNotFoundError(
        "Could not locate the fractional_pareto project root. "
        "Set FPDE_PROJECT_ROOT=/path/to/fractional_pareto_publication_ready_final "
        "or open the notebook from the project root/notebooks directory."
    )


ROOT = find_fpde_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")


from argparse import Namespace
import pandas as pd
from scripts.run_all_methods import run_all_methods

# EDIT THESE VALUES
dataset_names = ["synthetic_time_space_fractional_RD"]
noise_levels = [0]
seeds = [0]
profile = "notebook"  # use "paper" for final-quality runs
methods = ["weak_pareto", "vanilla_pareto"]  # add weak_grid_stridge/weak_fixed_stability for full comparisons

# Candidate-library overrides. In notebook mode we use a small teaching library;
# in paper mode, leave them as None to use the canonical paper library.
cmax_override = 2 if profile == "notebook" else None
p_values_override = (0,) if profile == "notebook" else None

# Runtime controls. In paper mode, leave maxiter/popsize as None.
maxiter_override = 0 if profile == "notebook" else None
popsize_override = 2 if profile == "notebook" else None
weak_test_budget = "smoke" if profile == "notebook" else "paper"
stability_splits = 1 if profile == "notebook" else 5
stability_width_scales = [1.0] if profile == "notebook" else [0.8, 1.0, 1.2]

OUT = ROOT / "results" / "notebook_method_comparison"
OUT.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)
print("Output dir:", OUT)


## 1. Build the exact script command

This cell constructs the same command you would run in a terminal. The notebook does not implement a separate benchmark path.


In [ ]:
cmd = [
    "python", "scripts/run_all_methods.py",
    "--datasets", *dataset_names,
    "--methods", *methods,
    "--profile", profile,
    "--noise-levels", *[str(v) for v in noise_levels],
    "--seeds", *[str(v) for v in seeds],
    "--weak-test-budget", weak_test_budget,
    "--stability-splits", str(stability_splits),
    "--stability-width-scales", *[str(v) for v in stability_width_scales],
    "--quiet",
    "--output-dir", str(OUT),
]
if maxiter_override is not None:
    cmd += ["--maxiter", str(maxiter_override)]
if popsize_override is not None:
    cmd += ["--popsize", str(popsize_override)]
if cmax_override is not None:
    cmd += ["--cmax", str(cmax_override)]
if p_values_override is not None:
    cmd += ["--p-values", *[str(v) for v in p_values_override]]
print(" \
  ".join(cmd))

## 2. Run the comparison

For large `profile="paper"` runs this may take a while. Use the terminal script for unattended final runs.


In [ ]:
args = Namespace(
    data_dir=ROOT / "data",
    output_dir=OUT,
    datasets=dataset_names,
    methods=methods,
    noise_levels=noise_levels,
    seeds=seeds,
    profile=profile,
    maxiter=maxiter_override,
    popsize=popsize_override,
    cmax=cmax_override,
    p_values=p_values_override,
    weak_test_budget=weak_test_budget,
    stability_splits=stability_splits,
    stability_width_scales=list(stability_width_scales),
    quiet=True,
    progress=False,
    progress_de=False,
)
rows = run_all_methods(args)
print(f"Finished {len(rows)} notebook-dispatched runs.")

## 3. Inspect the result table

In [ ]:
df = pd.read_csv(OUT / "method_comparison.csv")
cols = [
    "dataset", "noise_percent", "seed", "method", "recovery_label",
    "full_structure_recovered", "rhs_f1", "alpha_abs_error",
    "max_matched_beta_abs_error", "max_coef_rel_error",
    "selected_equation",
]
df[[c for c in cols if c in df.columns]]

## 4. Final paper command

For the full benchmark, run this from the project root:

```bash
bash scripts/run_publication_benchmarks.sh
```

For a quick smoke test:

```bash
FPDE_QUICK=1 bash scripts/run_publication_benchmarks.sh
```
